# 02 — Baseline BERT pour la classification de toxicité (Jigsaw)

**Mini-projet IA Responsable — Phase 1 / 5**

Ce notebook entraîne et évalue un modèle de référence `bert-base-uncased` finement ajusté sur le corpus *Jigsaw Unintended Bias in Toxicity Classification*, en suivant strictement le protocole décrit dans `docs/ROADMAP.md`.

**Sortie attendue** : un modèle sauvegardé, des prédictions parquet réutilisables par les phases suivantes (équité in-processing, SHAP, robustesse), et un fichier `baseline_metrics.json` avec :
- ROC-AUC global et ECE (calibration),
- Subgroup / BPSN / BNSP AUC pour chaque identité ayant ≥ 500 exemples en test,
- score Jigsaw final (moyenne arithmétique de l'overall AUC et des 3 moyennes généralisées avec *p* = −5).

---

**Sommaire**

0. Setup et configuration
1. Chargement des données
2. Splits stratifiés (80/10/10)
3. Tokenisation et `Dataset` HuggingFace
4. Modèle et `Trainer` BCE
5. Entraînement
6. Inférence sur le test set
7. Performance et calibration
8. Métriques d'équité par identité
9. Visualisations
10. Sauvegarde des artefacts
11. Diagnostic et étapes suivantes


## 0. Setup & Configuration


In [ ]:
# Décommenter pour installer les dépendances (Colab/Kaggle).
# !pip -q install "transformers>=4.40,<5" "datasets>=2.19,<3" "accelerate>=0.30" \
#                 "scikit-learn>=1.4" "pandas>=2.2" "numpy>=2,<3" "matplotlib" "seaborn" \
#                 "pyarrow" "tqdm"


In [ ]:
import json
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.metrics import roc_auc_score

# Évite l'import des backends TF / Flax via transformers (peut être très lent).
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("USE_TORCH", "1")

from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
sns.set_palette("tab10")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Torch  : {torch.__version__}")


In [ ]:
# === Chemins ===
# Soit on positionne JIGSAW_DATA_DIR (variable d'env), soit on adapte ci-dessous.
# Par défaut : on cherche le dataset dans projet/jigsaw-unintended-bias-in-toxicity-classification/
DATA_DIR = Path(
    os.environ.get(
        "JIGSAW_DATA_DIR",
        "../../jigsaw-unintended-bias-in-toxicity-classification",
    )
).expanduser().resolve()
TRAIN_CSV = DATA_DIR / "train.csv"

REPORTS_DIR     = Path("../reports").resolve()
CHECKPOINTS_DIR = REPORTS_DIR / "checkpoints" / "baseline_bert"
SPLITS_DIR      = REPORTS_DIR / "splits"
METRICS_DIR     = REPORTS_DIR / "metrics"
FIGURES_DIR     = REPORTS_DIR / "figures"
for p in (REPORTS_DIR, CHECKPOINTS_DIR, SPLITS_DIR, METRICS_DIR, FIGURES_DIR):
    p.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR        : {DATA_DIR}")
print(f"TRAIN_CSV       : {TRAIN_CSV} | exists = {TRAIN_CSV.exists()}")
print(f"CHECKPOINTS_DIR : {CHECKPOINTS_DIR}")

# === Hyperparamètres ===
SEED               = 1337
MODEL_NAME         = "bert-base-uncased"
MAX_LEN            = 128
BATCH_SIZE_TRAIN   = 32          # adapter à la VRAM (16 si <8 GB)
BATCH_SIZE_EVAL    = 64
LR                 = 2e-5
N_EPOCHS           = 2
WARMUP_RATIO       = 0.06
WEIGHT_DECAY       = 0.01
GRAD_ACCUM_STEPS   = 1
FP16               = True

# === Sous-échantillonnage (smoke vs full) ===
# None  -> dataset complet (~1.8 M lignes ; 2-3 h sur RTX 4080)
# 300_000 -> itération rapide en ~15-20 min, suffisant pour valider le pipeline
N_ROWS = 300_000

# === Splits ===
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.8, 0.1, 0.1
LABEL_THRESHOLD       = 0.5
IDENTITY_THRESHOLD    = 0.5
MIN_TEST_PER_IDENTITY = 500


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"Seed : {SEED}")


## 1. Chargement des données

On ne charge que les colonnes utiles (id, texte, target, 24 colonnes d'identités). Cela divise par ~5 la consommation mémoire vs `pd.read_csv` complet.


In [ ]:
IDENTITY_COLS = [
    "asian", "atheist", "bisexual", "black", "buddhist", "christian", "female",
    "heterosexual", "hindu", "homosexual_gay_or_lesbian",
    "intellectual_or_learning_disability", "jewish", "latino", "male", "muslim",
    "other_disability", "other_gender", "other_race_or_ethnicity", "other_religion",
    "other_sexual_orientation", "physical_disability", "psychiatric_or_mental_illness",
    "transgender", "white",
]
USE_COLS = ["id", "comment_text", "target"] + IDENTITY_COLS


In [ ]:
assert TRAIN_CSV.exists(), (
    f"Fichier introuvable: {TRAIN_CSV}\n"
    f"Définir JIGSAW_DATA_DIR ou modifier DATA_DIR dans la cellule de config."
)

print(f"Lecture de {TRAIN_CSV} …")
df = pd.read_csv(TRAIN_CSV, usecols=USE_COLS)
print(f"Total lignes : {len(df):,}")

if N_ROWS is not None and N_ROWS < len(df):
    df = df.sample(n=N_ROWS, random_state=SEED).reset_index(drop=True)
    print(f"Sous-échantillonnage à : {len(df):,} lignes (seed={SEED})")


In [ ]:
# Étiquette toxique binaire
df["label"] = (df["target"].fillna(0) >= LABEL_THRESHOLD).astype(int)

# Mention d'identité (binaire) + flag any_identity
for c in IDENTITY_COLS:
    df[c + "_bin"] = (df[c].fillna(0) >= IDENTITY_THRESHOLD).astype(int)
df["any_identity"] = df[[c + "_bin" for c in IDENTITY_COLS]].max(axis=1)

# Clé de stratification 4-classes (label × any_identity)
df["stratify_key"] = (df["label"] * 2 + df["any_identity"]).astype(int)

print(f"Taux toxique global    : {df['label'].mean():.4f}")
print(f"Taux any_identity      : {df['any_identity'].mean():.4f}")
print(f"\nDistribution stratify_key :")
print(df["stratify_key"].value_counts().sort_index().rename({
    0: "non-toxique × sans-identité",
    1: "non-toxique × identité",
    2: "toxique     × sans-identité",
    3: "toxique     × identité",
}))


## 2. Splits stratifiés (80 / 10 / 10)

Stratification par la clé `2 * label + any_identity` : on conserve l'identique le taux de toxicité **et** la fraction de commentaires mentionnant une identité dans les trois ensembles. C'est essentiel pour que les métriques d'équité calculées sur le test soient comparables entre modèles.


In [ ]:
def stratified_split(
    d: pd.DataFrame, *, seed: int,
    train_frac: float, val_frac: float, test_frac: float,
    stratify_col: str = "stratify_key",
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    assert abs(train_frac + val_frac + test_frac - 1.0) < 1e-6, "Les fractions doivent sommer à 1"
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for k, sub in d.groupby(stratify_col, sort=True):
        idx = sub.index.to_numpy()
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(n * train_frac)
        n_val   = int(n * val_frac)
        train_idx.extend(idx[:n_train].tolist())
        val_idx.extend(idx[n_train:n_train + n_val].tolist())
        test_idx.extend(idx[n_train + n_val:].tolist())
    rng.shuffle(train_idx); rng.shuffle(val_idx); rng.shuffle(test_idx)
    return (d.loc[train_idx].reset_index(drop=True),
            d.loc[val_idx].reset_index(drop=True),
            d.loc[test_idx].reset_index(drop=True))

train_df, val_df, test_df = stratified_split(
    df, seed=SEED,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
)

print(f"Train : {len(train_df):>9,} | Val : {len(val_df):>8,} | Test : {len(test_df):>8,}")
print("\nTaux par split :")
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"  {name:5s} | toxic={d['label'].mean():.4f} | any_identity={d['any_identity'].mean():.4f}")

# Persister les IDs (pour réutilisation par les phases équité/XAI/robustesse)
splits_payload = {
    "train": train_df["id"].astype(int).tolist(),
    "val":   val_df["id"].astype(int).tolist(),
    "test":  test_df["id"].astype(int).tolist(),
    "seed":  SEED,
    "n_rows_used": len(df),
}
with open(SPLITS_DIR / "split_ids.json", "w", encoding="utf-8") as f:
    json.dump(splits_payload, f)
print(f"\nIDs sauvegardés : {SPLITS_DIR / 'split_ids.json'}")


## 3. Tokenisation et `Dataset` HuggingFace

Tronquage à `MAX_LEN = 128` (la majorité des commentaires Civil Comments fait moins de 100 tokens). Padding à `max_length` pour simplifier le `DataCollator`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def to_hf_ds(d: pd.DataFrame) -> Dataset:
    return Dataset.from_pandas(
        d[["comment_text", "label"]].rename(columns={"comment_text": "text"}),
        preserve_index=False,
    )

dsd = DatasetDict(train=to_hf_ds(train_df), val=to_hf_ds(val_df), test=to_hf_ds(test_df))

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True, padding="max_length", max_length=MAX_LEN,
    )

print("Tokenisation …")
dsd_tok = dsd.map(tokenize, batched=True, remove_columns=["text"], desc="tokenize")
dsd_tok.set_format("torch")
print(dsd_tok)


## 4. Modèle et `Trainer` BCE

`AutoModelForSequenceClassification` avec `num_labels = 1` produit un seul logit par exemple. On utilise `BCEWithLogitsLoss` (numériquement plus stable que `sigmoid + BCE`). Le `Trainer` standard de HuggingFace utilise par défaut `CrossEntropyLoss` ; on en sous-classe une version qui calcule la BCE.


In [ ]:
class BCEWithLogitsTrainer(Trainer):
    """Trainer 1-logit + BCEWithLogitsLoss. Compatible Transformers >=4.40."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float()
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(-1)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels)
        return (loss, outputs) if return_outputs else loss


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.asarray(logits).reshape(-1)
    labels = np.asarray(labels).reshape(-1)
    probs = sigmoid(logits)
    return {"roc_auc": float(roc_auc_score(labels, probs))}


model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
n_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres modèle : {n_params/1e6:.1f} M")


## 5. Entraînement

⏱️ **Ordre de grandeur** : avec `N_ROWS = 300_000`, batch 32, max_len 128, fp16, RTX 4080 ⇒ ~15-20 min pour 2 epochs.
Pour le run de référence sur le dataset complet : `N_ROWS = None`, ~2-3 h.


In [ ]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE_TRAIN,
    per_device_eval_batch_size=BATCH_SIZE_EVAL,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    fp16=FP16 and torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    seed=SEED,
    report_to=[],   # désactive WandB / TensorBoard par défaut
)

trainer = BCEWithLogitsTrainer(
    model=model,
    args=training_args,
    train_dataset=dsd_tok["train"],
    eval_dataset=dsd_tok["val"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Démarrage de l'entraînement …")
train_result = trainer.train()
print("\nMétriques d'entraînement :")
for k, v in train_result.metrics.items():
    print(f"  {k:<28s} {v}")


In [ ]:
# Persiste le meilleur modèle (chargé automatiquement par load_best_model_at_end=True).
trainer.save_model(str(CHECKPOINTS_DIR))
tokenizer.save_pretrained(str(CHECKPOINTS_DIR))
print(f"Modèle sauvegardé : {CHECKPOINTS_DIR}")


## 6. Inférence sur le test set


In [ ]:
@torch.inference_mode()
def predict_proba(model, tokenizer, texts, *, max_length, batch_size=64, device=DEVICE):
    model.to(device).eval()
    all_logits = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i + batch_size],
            truncation=True, padding=True, max_length=max_length, return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        all_logits.append(out.logits.squeeze(-1).detach().cpu().numpy())
    logits = np.concatenate(all_logits, axis=0)
    return sigmoid(logits), logits


texts_test = test_df["comment_text"].astype(str).tolist()
y_true = test_df["label"].to_numpy()

print(f"Inférence sur {len(texts_test):,} commentaires …")
probs_test, logits_test = predict_proba(
    model, tokenizer, texts_test,
    max_length=MAX_LEN, batch_size=BATCH_SIZE_EVAL,
)

# Stocke prédictions pour les phases suivantes (XAI, robustesse, équité)
test_predictions = test_df.copy()
test_predictions["pred_prob"]  = probs_test
test_predictions["pred_logit"] = logits_test
pred_path = METRICS_DIR / "test_predictions.parquet"
test_predictions.to_parquet(pred_path, index=False)
print(f"Prédictions écrites : {pred_path}")


## 7. Performance globale et calibration

- **ROC-AUC** : performance globale, indépendante du seuil.
- **ECE** (Expected Calibration Error, 15 bins) : un modèle bien calibré a une confiance moyenne par bin égale à la précision observée. ECE ↑ ⇒ surconfiance (typique sous attaque adverse).


In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins: int = 15) -> float:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= 0.5).astype(int)
    conf = np.where(y_pred == 1, y_prob, 1.0 - y_prob)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece, n = 0.0, len(y_true)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf < hi if i < n_bins - 1 else conf <= hi)
        if not np.any(mask):
            continue
        acc = float(np.mean(y_pred[mask] == y_true[mask]))
        avg_conf = float(np.mean(conf[mask]))
        ece += (np.sum(mask) / n) * abs(acc - avg_conf)
    return float(ece)


overall_auc = float(roc_auc_score(y_true, probs_test))
ece         = expected_calibration_error(y_true, probs_test, n_bins=15)
y_pred_bin  = (probs_test >= 0.5).astype(int)
accuracy    = float(np.mean(y_pred_bin == y_true))

print(f"Overall ROC-AUC : {overall_auc:.4f}")
print(f"ECE (15 bins)   : {ece:.4f}")
print(f"Accuracy (>=.5) : {accuracy:.4f}")


## 8. Métriques d'équité par identité

Pour chaque identité avec au moins `MIN_TEST_PER_IDENTITY = 500` exemples dans le test, on calcule (Borkan et al., 2019) :

| Métrique | Pos | Neg | Mesure |
|---|---|---|---|
| **Subgroup AUC** | toxiques mentionnant l'identité | non-toxiques mentionnant l'identité | séparabilité **dans** le sous-groupe |
| **BPSN AUC** (Background Pos, Subgroup Neg) | toxiques sans identité | **non-toxiques** mentionnant l'identité | propension aux **faux positifs** identitaires |
| **BNSP AUC** (Background Neg, Subgroup Pos) | **toxiques** mentionnant l'identité | non-toxiques sans identité | propension aux **faux négatifs** identitaires |

AUC < 0.5 ⇔ biais systémique. On agrège ces AUC via la **moyenne généralisée** $M_p(x) = (\frac{1}{N}\sum x_i^p)^{1/p}$ avec $p = -5$ pour pénaliser fortement les pires sous-groupes.


In [ ]:
def safe_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(int)
    if y_true.size == 0 or y_true.min() == y_true.max():
        return float("nan")
    return float(roc_auc_score(y_true, y_score))


def compute_identity_aucs(y_true: np.ndarray, y_pred: np.ndarray, sub_mask: np.ndarray) -> dict:
    sub = sub_mask.astype(bool)
    bg  = ~sub
    sg = safe_auc(y_true[sub], y_pred[sub])
    bpsn_mask = (sub & (y_true == 0)) | (bg & (y_true == 1))
    bnsp_mask = (sub & (y_true == 1)) | (bg & (y_true == 0))
    return {
        "subgroup_auc": sg,
        "bpsn_auc":     safe_auc(y_true[bpsn_mask], y_pred[bpsn_mask]),
        "bnsp_auc":     safe_auc(y_true[bnsp_mask], y_pred[bnsp_mask]),
        "n_subgroup":   int(sub.sum()),
    }


records = []
for col in IDENTITY_COLS:
    sub_mask = test_df[col + "_bin"].to_numpy().astype(bool)
    n_sub = int(sub_mask.sum())
    if n_sub < MIN_TEST_PER_IDENTITY:
        continue
    aucs = compute_identity_aucs(y_true, probs_test, sub_mask)
    records.append({"identity": col, **aucs})

fairness_df = pd.DataFrame(records).sort_values("subgroup_auc").reset_index(drop=True)
print(f"Identités retenues (n_test ≥ {MIN_TEST_PER_IDENTITY}) : {len(fairness_df)}\n")
print(fairness_df.to_string(index=False))


In [ ]:
def generalized_mean(values, p: float = -5.0) -> float:
    arr = np.asarray([v for v in values if np.isfinite(v)], dtype=float)
    if arr.size == 0:
        return float("nan")
    return float(np.power(np.mean(np.power(arr, p)), 1.0 / p))


P_GEN_MEAN = -5.0
gm_sub  = generalized_mean(fairness_df["subgroup_auc"], p=P_GEN_MEAN)
gm_bpsn = generalized_mean(fairness_df["bpsn_auc"],     p=P_GEN_MEAN)
gm_bnsp = generalized_mean(fairness_df["bnsp_auc"],     p=P_GEN_MEAN)
final_score = float(np.nanmean([overall_auc, gm_sub, gm_bpsn, gm_bnsp]))

print(f"Moyennes généralisées (p = {P_GEN_MEAN}) :")
print(f"  Subgroup : {gm_sub:.4f}")
print(f"  BPSN     : {gm_bpsn:.4f}")
print(f"  BNSP     : {gm_bnsp:.4f}")
print(f"\nOverall ROC-AUC      : {overall_auc:.4f}")
print(f"Jigsaw final score   : {final_score:.4f}")
print(f"  = (overall + gm_sub + gm_bpsn + gm_bnsp) / 4")


## 9. Visualisations


In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, 0.32 * len(fairness_df))))
df_plot = fairness_df.set_index("identity")[["subgroup_auc", "bpsn_auc", "bnsp_auc"]].copy()
df_plot = df_plot.sort_values("subgroup_auc")
y = np.arange(len(df_plot))
w = 0.27
ax.barh(y - w, df_plot["subgroup_auc"], w, label="Subgroup AUC", color="#4C72B0", edgecolor="white", lw=0.4)
ax.barh(y,     df_plot["bpsn_auc"],     w, label="BPSN AUC",     color="#DD8452", edgecolor="white", lw=0.4)
ax.barh(y + w, df_plot["bnsp_auc"],     w, label="BNSP AUC",     color="#55A868", edgecolor="white", lw=0.4)
ax.axvline(overall_auc, color="black", lw=1.0, ls="--", label=f"Overall AUC = {overall_auc:.3f}")
ax.axvline(0.5, color="red", lw=0.7, ls=":", label="AUC = 0.5 (aléatoire)")
ax.set_yticks(y); ax.set_yticklabels(df_plot.index)
ax.set_xlim(0.5, 1.0); ax.set_xlabel("AUC")
ax.set_title("Métriques d'équité par identité — BERT baseline")
ax.legend(loc="lower left", fontsize=9, ncol=2)
plt.tight_layout()
fig_path = FIGURES_DIR / "baseline_per_identity_aucs.png"
plt.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"Figure : {fig_path}")


In [ ]:
heat = fairness_df.set_index("identity")[["subgroup_auc", "bpsn_auc", "bnsp_auc"]].copy()
heat.columns = ["Subgroup", "BPSN", "BNSP"]
heat = heat.sort_values("Subgroup")

fig, ax = plt.subplots(figsize=(6, max(4, 0.30 * len(heat))))
sns.heatmap(
    heat, annot=True, fmt=".3f",
    cmap="RdYlGn", vmin=0.5, vmax=1.0, center=0.85,
    ax=ax, cbar_kws={"label": "AUC (rouge = biais)"},
    linewidths=0.4,
)
ax.set_title("AUC d'équité par identité\n(plus rouge = plus biaisé)")
ax.set_ylabel("")
plt.tight_layout()
fig_path = FIGURES_DIR / "baseline_auc_heatmap.png"
plt.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"Figure : {fig_path}")


In [ ]:
# Diagramme de fiabilité (calibration)
n_bins = 15
y_true_bin = y_true.astype(int)
y_pred = (probs_test >= 0.5).astype(int)
conf = np.where(y_pred == 1, probs_test, 1 - probs_test)
bins = np.linspace(0, 1, n_bins + 1)
bin_acc, bin_conf, bin_n = [], [], []
for i in range(n_bins):
    lo, hi = bins[i], bins[i + 1]
    mask = (conf >= lo) & (conf < hi if i < n_bins - 1 else conf <= hi)
    if mask.sum() == 0:
        bin_acc.append(np.nan); bin_conf.append(np.nan); bin_n.append(0)
    else:
        bin_acc.append(float((y_pred[mask] == y_true_bin[mask]).mean()))
        bin_conf.append(float(conf[mask].mean()))
        bin_n.append(int(mask.sum()))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="parfaitement calibré")
ax.plot(bin_conf, bin_acc, marker="o", lw=1.5, color="#4C72B0", label="modèle")
ax.set_xlabel("Confiance moyenne par bin")
ax.set_ylabel("Précision observée par bin")
ax.set_title(f"Diagramme de fiabilité — ECE = {ece:.4f}")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
fig_path = FIGURES_DIR / "baseline_reliability.png"
plt.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"Figure : {fig_path}")


## 10. Sauvegarde des artefacts


In [ ]:
results = {
    "model_name": MODEL_NAME,
    "seed": SEED,
    "n_rows_used": int(len(df)),
    "n_train": int(len(train_df)), "n_val": int(len(val_df)), "n_test": int(len(test_df)),
    "max_length": MAX_LEN,
    "epochs": N_EPOCHS,
    "batch_size_train": BATCH_SIZE_TRAIN,
    "learning_rate": LR,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "fp16": bool(FP16 and torch.cuda.is_available()),
    "overall_auc": float(overall_auc),
    "accuracy_at_0.5": float(accuracy),
    "ece_15bins": float(ece),
    "jigsaw_final_score": float(final_score),
    "generalized_mean_p": P_GEN_MEAN,
    "generalized_mean": {
        "subgroup_auc": float(gm_sub),
        "bpsn_auc":     float(gm_bpsn),
        "bnsp_auc":     float(gm_bnsp),
    },
    "min_test_per_identity": MIN_TEST_PER_IDENTITY,
    "n_identities_used": int(len(fairness_df)),
    "per_identity": fairness_df.set_index("identity").to_dict(orient="index"),
}

metrics_path = METRICS_DIR / "baseline_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print(f"Métriques sauvegardées : {metrics_path}")

# Récapitulatif lisible
print("\n=== RÉCAPITULATIF BASELINE BERT ===")
print(f"  N rows                : {results['n_rows_used']:,}")
print(f"  Overall AUC           : {results['overall_auc']:.4f}")
print(f"  ECE (15 bins)         : {results['ece_15bins']:.4f}")
print(f"  Jigsaw final score    : {results['jigsaw_final_score']:.4f}")
print(f"  Identités évaluées    : {results['n_identities_used']}")
print(f"  Min Subgroup AUC      : {fairness_df['subgroup_auc'].min():.4f}  ({fairness_df.iloc[0]['identity']})")
print(f"  Min BPSN AUC          : {fairness_df['bpsn_auc'].min():.4f}  ({fairness_df.loc[fairness_df['bpsn_auc'].idxmin(), 'identity']})")
print(f"  Min BNSP AUC          : {fairness_df['bnsp_auc'].min():.4f}  ({fairness_df.loc[fairness_df['bnsp_auc'].idxmin(), 'identity']})")


## 11. Diagnostic et étapes suivantes

### Lecture des résultats baseline

Si les hypothèses de la littérature se confirment, on s'attend à :

- **Overall AUC** entre **0.92 et 0.96** (selon `N_ROWS`).
- **Subgroup / BPSN / BNSP AUC** systématiquement plus faibles que l'overall AUC, particulièrement pour les identités historiquement sur-représentées dans des contextes hostiles : `homosexual_gay_or_lesbian`, `muslim`, `black`, `transgender`, `psychiatric_or_mental_illness`.
- **Score Jigsaw final** typiquement **0.04 à 0.08 points en dessous de l'overall AUC** — la pénalité de moyenne généralisée révèle l'asymétrie du modèle.

### Outputs réutilisables

| Artefact | Réutilisé en phase |
|---|---|
| `reports/checkpoints/baseline_bert/` | P3 (SHAP), P4 (TextAttack) |
| `reports/metrics/test_predictions.parquet` | P5 (synthèse comparative) |
| `reports/splits/split_ids.json` | **P2 (équité)** ⇒ fixe le test set pour comparer modèles |
| `reports/metrics/baseline_metrics.json` | P5 |

### Prochaine étape : Phase 2 — Mitigation équité in-processing

→ Notebook `03_fairness_inprocessing.ipynb` :
1. Recharger **les mêmes splits** (`split_ids.json`) — comparabilité absolue.
2. Variante A : `BCEWithLogitsTrainer` + `loss_weight = 1 + 0.25 * is_BPSN_or_BNSP`.
3. Variante B : `BertForToxicityAndIdentity` + `MultiTaskBCETrainer` (`α = 0.1`).
4. Recalculer **exactement** les mêmes métriques que ci-dessus, sur le **même** test set.
5. Tableau comparatif Δoverall_auc, Δjigsaw_score, Δ min(BPSN_AUC) — c'est là que se lit le compromis perf./équité.
